# 02 · Preprocesamiento y partición temporal
Prepara los datos para el modelado. Los criterios son estos:
- **Partición temporal:** C1–C3 entrenamiento, C4 selección y C5 prueba. No hay mezcla aleatoria entre años. **C1 es condicional:** entra solo si supera la auditoría de comparabilidad (sección 4).
- **Guardas anti-fuga:** la matriz X solo contiene variables conocidas antes del 20 de febrero.
- **Transformaciones:** se ajustan **solo** con el entrenamiento.
- **Desbalance:** se trata con `class_weight="balanced"`, sin sobremuestreo, para no fabricar estudiantes artificiales en una muestra pequeña.

**Convención de la etiqueta:** `y_no_matricula = 1` significa que el estudiante *no* pagó la matrícula entre el 20-feb y el 30-abr (evento de interés). Equivale a `1 − matricula_efectiva` del documento v5.

In [ ]:
# @title 0 · Preparar el entorno (ejecutar primero)
# Si el repositorio ya está en GitHub, pegue su URL; si no, se pedirá subir el .zip.
REPO_URL = ""  # @param {type:"string"}

import importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/prediccion-matricula-lemas")
    if not RAIZ.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Suba el archivo prediccion-matricula-lemas.zip")
            nombre = next(iter(files.upload()))
            zipfile.ZipFile(nombre).extractall("/content")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
REQUERIDOS = {'pandas': 'pandas', 'sklearn': 'scikit-learn', 'yaml': 'PyYAML'}  # módulo -> paquete pip
faltan = [pip for modulo, pip in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Raíz:", RAIZ)

In [ ]:
# @title Cargar la base
FUENTE = "sintetica"  # @param ["sintetica", "real"]
# Marque SUBIR_DE_NUEVO si subió un archivo equivocado (se borra y se vuelve a pedir)
SUBIR_DE_NUEVO = False  # @param {type:"boolean"}

import pandas as pd
from src.data_processing import construir_dataset, limpiar_base
from src.utils import cargar_config, configurar_logging, obtener_base_seud

configurar_logging()
config = cargar_config()
MIN = config["privacidad"]["min_celda"]
ruta = RAIZ / config["rutas"]["base_real" if FUENTE == "real" else "base_sintetica"]

crudo = (obtener_base_seud(ruta, EN_COLAB, SUBIR_DE_NUEVO) if FUENTE == "real"
         else pd.read_csv(ruta, dtype=str))
base = limpiar_base(crudo, config)
dataset, reporte = construir_dataset(base, config)
if dataset.empty or "y_no_matricula" not in dataset:
    anios = crudo["anoa"].astype(str).str.strip().value_counts().head(10).to_dict()
    raise RuntimeError(
        "No se pudo construir ninguna cohorte. Revise la columna anoa: debe tener el año "
        f"de inicio del ciclo (2021…2026). Valores más frecuentes: {anios}")
print(f"Fuente: {FUENTE} | filas base: {len(base)} | filas modelables: {len(dataset)}")
if FUENTE == "sintetica":
    print("⚠️ Datos SINTÉTICOS: sirven para probar el código, no describen a LEMAS.")

## 1. Partición temporal por cohortes

In [ ]:
from src.data_processing import CATEGORICAS, NUMERICAS, construir_matriz_x, verificar_sin_fuga

particiones = {rol: dataset[dataset["rol"] == rol] for rol in ["entrenamiento", "seleccion", "prueba"]}
resumen = pd.DataFrame({rol: {"cohortes": ", ".join(sorted(d["cohorte"].unique())), "estudiantes": len(d),
                              "representantes": d["id_familia"].nunique(),
                              "no_matricula": int(d["y_no_matricula"].sum()),
                              "% no_matricula": round(100 * d["y_no_matricula"].mean(), 1)}
                        for rol, d in particiones.items()}).T
display(resumen)
familias_train = set(particiones["entrenamiento"]["id_familia"])
repetidas = particiones["prueba"]["id_familia"].isin(familias_train).mean()
print(f"Representantes de C5 que también aparecen en entrenamiento: {100 * repetidas:.1f}% "
      "(esperado: las familias continúan año a año; la validación es temporal, no por familia).")

## 2. Matriz de predictores y verificación anti-fuga

In [ ]:
X = {rol: construir_matriz_x(d) for rol, d in particiones.items()}
y = {rol: d["y_no_matricula"].to_numpy() for rol, d in particiones.items()}
for rol, matriz in X.items():
    verificar_sin_fuga(matriz, config)
print("✅ Sin fuga. Predictores:", CATEGORICAS + NUMERICAS)
display(X["entrenamiento"].describe(include="all").T)

## 3. Transformaciones ajustadas solo con entrenamiento

In [ ]:
import numpy as np
from src.data_processing import construir_preprocesador

prep = construir_preprocesador(escalar=True).fit(X["entrenamiento"])
transformadas = {rol: prep.transform(m) for rol, m in X.items()}
nombres = prep.get_feature_names_out()
print(f"{len(nombres)} columnas tras la transformación")
for rol, m in transformadas.items():
    print(f"  {rol:13s} {m.shape}  NaN: {int(np.isnan(m).sum())}")

for col in CATEGORICAS:
    vistas = set(X["entrenamiento"][col])
    nuevas = {rol: sorted(set(X[rol][col]) - vistas) for rol in ("seleccion", "prueba")}
    if any(nuevas.values()):
        print(f"Categorías nuevas en {col}: {nuevas} → se agrupan como infrecuentes")

## 4. Comparabilidad de C1: entrenamiento con y sin C1 evaluado en C4
Cumple la Ficha técnica (26-sep-2026). Se entrena la **regresión logística de referencia** con `C1+C2+C3` y con `C2+C3`, y ambas se evalúan en **C4** con la métrica principal: Precision@k y Lift@k por representante, con k por sede según la regla aprobada. **C5 no se toca.**

Regla de decisión: C1 se incluye si tiene eventos suficientes y no reduce la Precision@k en C4 más que `auditoria.tolerancia_precision_k`. Esta comparación es preliminar; el ajuste fino de modelos es de la Fase 2.

In [ ]:
from src.auditoria import comparar_configuraciones, conteo_eventos, decidir_inclusion_c1
from src.evaluate import verificar_capacidad

sin_prueba = dataset[dataset["rol"] != "prueba"]          # garantía explícita: C5 fuera
comparacion = comparar_configuraciones(sin_prueba, config)
display(comparacion.drop(columns="k_por_sede"))

INCLUIR_C1, motivo = decidir_inclusion_c1(comparacion, conteo_eventos(dataset, config), config)
print(("✅ " if INCLUIR_C1 else "⚠️ ") + motivo)

k_elegido = comparacion.set_index("configuracion").loc["con_C1" if INCLUIR_C1 else "sin_C1", "k_por_sede"]
print("k por sede (regla aprobada):", k_elegido)
display(verificar_capacidad(k_elegido, config))

In [ ]:
# @title Registrar la decisión (se copia a docs/planificacion.md)
import json
decision = {"fuente": FUENTE, "incluir_C1": bool(INCLUIR_C1), "motivo": motivo,
            "k_por_sede": k_elegido,
            "comparacion": comparacion.drop(columns="k_por_sede").to_dict(orient="records")}
ruta_decision = RAIZ / config["rutas"]["metricas"] / f"decision_c1_{FUENTE}.json"
ruta_decision.parent.mkdir(parents=True, exist_ok=True)
ruta_decision.write_text(json.dumps(decision, indent=2, ensure_ascii=False), encoding="utf-8")
print("Guardado:", ruta_decision, "(solo métricas agregadas)")
if not INCLUIR_C1:
    dataset.loc[dataset["cohorte"] == "C1", "rol"] = "excluida"
    print("C1 marcada como 'excluida' en la tabla de modelado.")

## 5. Reporte obligatorio por cohorte (v5, sección 6.3)
Tabla del documento con N, M, U, q, familias, q familiar, exclusiones y comparabilidad. Con datos reales se ocultan los conteos menores a 5 antes de guardarla o compartirla.

In [ ]:
from src.auditoria import reporte_por_cohorte, suprimir_reporte

tabla_6_3 = reporte_por_cohorte(dataset, reporte, config, incluir_c1=INCLUIR_C1)
publica = suprimir_reporte(tabla_6_3, MIN) if FUENTE == "real" else tabla_6_3
display(publica)
ruta_6_3 = RAIZ / config["rutas"]["metricas"] / f"reporte_6_3_{FUENTE}.csv"
publica.to_csv(ruta_6_3, index=False)
print("Guardado:", ruta_6_3)

## 6. Análisis de sensibilidad: solo reservas en estado "Aprobar"
Como el sistema no registra la fecha de aprobación, se repite la construcción de la población excluyendo "Aprobar extraordinaria" (v5, secciones 5.3.1 y 5.3.2). Si la tasa de no matrícula cambia mucho, las aprobaciones tardías están influyendo en los resultados.

In [ ]:
from src.auditoria import sensibilidad_estados

sensibilidad = sensibilidad_estados(base, config)
if FUENTE == "real":
    for col in [c for c in sensibilidad.columns if c.startswith("estudiantes_")]:
        sensibilidad[col] = sensibilidad[col].astype(object).where(sensibilidad[col] >= MIN, f"<{MIN}")
display(sensibilidad)

## 7. Guardar la tabla de modelado

In [ ]:
salida = RAIZ / config["rutas"]["procesados"] / f"dataset_modelado_{FUENTE}.csv"
salida.parent.mkdir(parents=True, exist_ok=True)
dataset.drop(columns=["conducta_letra"], errors="ignore").to_csv(salida, index=False)
print("Guardado:", salida, "(data/processed/ no se sube al repositorio)")